In [ ]:
import numpy as np
import pandas as pd

#### load forecasts

In [ ]:
# load load forecasts
data_load_forecast = (
    pd.read_excel(
        "../../data/raw/smard/Prognostizierter_Stromverbrauch_201810010000_202501010000_Stunde.xlsx",
        skiprows=9,
        dtype=str,
    )
    .iloc[:, :3]
    .drop(columns=["Datum bis"])
)
data_load_forecast.columns = ["Date", "Load_forecast"]
data_load_forecast["Date"] = pd.to_datetime(data_load_forecast["Date"], dayfirst=True)
data_load_forecast["Load_forecast"] = (
    data_load_forecast["Load_forecast"].replace("-", np.nan).astype(float)
)
data_load_forecast

In [ ]:
# check for duplicated dates
duplicated_dates = data_load_forecast[
    data_load_forecast.duplicated(subset="Date", keep=False)
]["Date"].unique()
print(duplicated_dates)

# overwrite duplicated values with mean values
for date in duplicated_dates:
    mean_value = data_load_forecast[data_load_forecast["Date"] == date][
        "Load_forecast"
    ].mean()
    data_load_forecast.loc[data_load_forecast["Date"] == date, "Load_forecast"] = (
        mean_value
    )

# drop duplicate rows
data_load_forecast = data_load_forecast.drop_duplicates(
    subset="Date", keep="first"
).reset_index(drop=True)

#### generation renewable energy forecasts

In [ ]:
# load generation renewable energy forecasts
data_renewable_energy_forecast = (
    pd.read_excel(
        "../../data/raw/smard/Prognostizierte_Erzeugung_Day-Ahead_201810010000_202501010000_Stunde.xlsx",
        skiprows=9,
        dtype=str,
    )
    .iloc[:, [0, 1, 3]]
    .drop(columns=["Datum bis"])
)
data_renewable_energy_forecast.columns = ["Date", "Renewable_energy_forecast"]
data_renewable_energy_forecast["Date"] = pd.to_datetime(
    data_renewable_energy_forecast["Date"], dayfirst=True
)
data_renewable_energy_forecast["Renewable_energy_forecast"] = (
    data_renewable_energy_forecast["Renewable_energy_forecast"]
    .replace("-", np.nan)
    .astype(float)
)
data_renewable_energy_forecast

In [ ]:
# check for duplicated dates
duplicated_dates = data_renewable_energy_forecast[
    data_renewable_energy_forecast.duplicated(subset="Date", keep=False)
]["Date"].unique()
print(duplicated_dates)

# overwrite duplicated values with mean values
for date in duplicated_dates:
    mean_value = data_renewable_energy_forecast[
        data_renewable_energy_forecast["Date"] == date
    ]["Renewable_energy_forecast"].mean()
    data_renewable_energy_forecast.loc[
        data_renewable_energy_forecast["Date"] == date, "Renewable_energy_forecast"
    ] = mean_value

# drop duplicate rows
data_renewable_energy_forecast = data_renewable_energy_forecast.drop_duplicates(
    subset="Date", keep="first"
).reset_index(drop=True)

#### prices

In [ ]:
# load prices
data_price = (
    pd.read_excel(
        "../../data/raw/smard/Gro_handelspreise_201810010000_202501010000_Stunde.xlsx",
        skiprows=9,
        dtype=str,
    )
    .iloc[:, :3]
    .drop(columns=["Datum bis"])
)
data_price.columns = ["Date", "Price"]
data_price["Date"] = pd.to_datetime(data_price["Date"], dayfirst=True)
data_price["Price"] = data_price["Price"].replace("-", np.nan).astype(float)
data_price

In [ ]:
# check for duplicated dates
duplicated_dates = data_price[data_price.duplicated(subset="Date", keep=False)][
    "Date"
].unique()
print(duplicated_dates)

# overwrite duplicated values with mean values
for date in duplicated_dates:
    mean_value = data_price[data_price["Date"] == date]["Price"].mean()
    data_price.loc[data_price["Date"] == date, "Price"] = mean_value

# drop duplicate rows
data_price = data_price.drop_duplicates(subset="Date", keep="first").reset_index(
    drop=True
)

#### merge data to one file

In [ ]:
# Merge the datasets on the 'Date' column
merged_data = data_price.merge(data_load_forecast, on="Date", how="outer").merge(
    data_renewable_energy_forecast, on="Date", how="outer"
)
merged_data

#### add date column

In [ ]:
# add all dates and hours from 2018-10-01 to 2025-01-01
data = pd.DataFrame()
data["Date"] = pd.date_range(start="2018-10-01", end="2025-01-01", freq="h")
data = data.merge(merged_data, on="Date", how="left").reset_index(drop=True)
data

#### fill single None values

In [ ]:
# fill all alone nan values with the mean of the previous and next value
data["Price"] = data["Price"].fillna(
    (data["Price"].shift() + data["Price"].shift(-1)) / 2
)
data["Load_forecast"] = data["Load_forecast"].fillna(
    (data["Load_forecast"].shift() + data["Load_forecast"].shift(-1)) / 2
)
data["Renewable_energy_forecast"] = data["Renewable_energy_forecast"].fillna(
    (
        data["Renewable_energy_forecast"].shift()
        + data["Renewable_energy_forecast"].shift(-1)
    )
    / 2
)
data

In [ ]:
# save the data to excel file
data.to_excel(
    "../../data/processed/smard_data_combined_201810010000_202501010000.xlsx",
    index=False,
)

### construct dataset for training

In [ ]:
# merge all hours in one row for each day
data_features = data.groupby(
    data["Date"].dt.date,
).agg(
    {
        "Price": list,
        "Load_forecast": list,
        "Renewable_energy_forecast": list,
    }
)
data_features

In [ ]:
# if list contains nan values, replace list with None
data_features["Price"] = data_features["Price"].apply(
    lambda x: x if not any(pd.isna(x)) else None
)
data_features["Load_forecast"] = data_features["Load_forecast"].apply(
    lambda x: x if not any(pd.isna(x)) else None
)
data_features["Renewable_energy_forecast"] = data_features[
    "Renewable_energy_forecast"
].apply(lambda x: x if not any(pd.isna(x)) else None)
data_features

In [ ]:
# add column price shifted
data_features["Price d-1"] = data_features["Price"].shift(1)
data_features["Price d-2"] = data_features["Price"].shift(2)
data_features["Price d-3"] = data_features["Price"].shift(3)
data_features["Price d-7"] = data_features["Price"].shift(7)
data_features

In [ ]:
# add weekday dummies embeddings
def weekday_dummies(weekday):
    dummies = [0] * 7
    dummies[weekday] = 1
    return dummies


data_features["Weekday"] = pd.to_datetime(data_features.index).weekday
data_features["Weekday"] = data_features["Weekday"].apply(weekday_dummies)
data_features

In [ ]:
# add column to display if None values in row
data_features["None values"] = data_features.isna().any(axis=1)
data_features

In [ ]:
# save the data to excel file
data_features.to_excel(
    "../../data/processed/smard_dataset_with_Nones_201810010000_202501010000.xlsx"
)

In [ ]:
# drop rows with None values and drop the column 'None values'
data_features_without_None = data_features.dropna().drop(columns=["None values"])
data_features_without_None

In [ ]:
# concatenate all columns in one row and convert the dataset to a numpy array
dataset = data_features_without_None.apply(lambda x: np.concatenate(x.values), axis=1)
dataset = np.array(dataset.tolist()).astype(float)
print(dataset)
features, targets = dataset[:, 24:], dataset[:, :24]

In [ ]:
# save the data to npz file
np.savez(
    "../../data/processed/smard_data_201810010000_202501010000.npz",
    features=features,
    targets=targets,
    dates=pd.to_datetime(data_features_without_None.index),
)